In [1]:
from tensorflow.keras.datasets import mnist
import numpy as np
from numba import cuda
import math
import time

In [2]:
(X_train, y_train), (_, _) = mnist.load_data()

# Select first image
image = X_train[0].astype(np.float32)

# Flatten to 784 values
data = image.flatten()

print("Input Length:", len(data))
print("Label:", y_train[0])

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Input Length: 784
Label: 5


In [3]:
@cuda.jit
def inclusive_scan_kernel(input_arr, output_arr, stride):

    idx = cuda.grid(1)

    if idx < input_arr.size:

        if idx >= stride:
            output_arr[idx] = input_arr[idx] + input_arr[idx - stride]
        else:
            output_arr[idx] = input_arr[idx]

In [4]:
def gpu_inclusive_scan(arr):

    n = len(arr)

    d_current = cuda.to_device(arr)

    threads = 256
    blocks = math.ceil(n / threads)

    stride = 1

    while stride < n:

        d_next = cuda.device_array_like(d_current)

        inclusive_scan_kernel[blocks, threads](
            d_current,
            d_next,
            stride
        )

        d_current = d_next

        stride *= 2

    return d_current.copy_to_host()

In [5]:
start_gpu = time.time()

gpu_result = gpu_inclusive_scan(data)

cuda.synchronize()

gpu_time = time.time() - start_gpu

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [6]:
start_cpu = time.time()

cpu_result = np.cumsum(data)

cpu_time = time.time() - start_cpu

In [7]:
print("GPU Time :", gpu_time)
print("CPU Time :", cpu_time)

print("Results Match:",
      np.allclose(gpu_result, cpu_result))

print("\nFirst 20 Values:")
print(gpu_result[:20])

GPU Time : 2.423447608947754
CPU Time : 0.00021719932556152344
Results Match: True

First 20 Values:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
